# 5.1

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [ ]:
# Set plotting style
sns.set_theme(style="whitegrid")

# Load + plot data for multiple rhos in a 2x2 grid
# We assume the notebook is located in 'QueueTorch/notebooks' and results in 'QueueTorch/results'
rhos = [0.8, 0.9, 0.95, 0.99]
file_template = "../results/gradient_comparison_criss_cross_bh_rho{rho}.json"

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
axes = axes.flatten()

for ax, rho in zip(axes, rhos):
    results_path = file_template.format(rho=rho)

    if not os.path.exists(results_path):
        ax.axis('off')
        ax.set_title(f"rho={rho} (fichier introuvable)")
        continue

    with open(results_path, 'r') as f:
        data = json.load(f)

    df = pd.DataFrame(data).dropna()

    if df.empty:
        ax.axis('off')
        ax.set_title(f"rho={rho} (pas de données valides)")
        continue

    # Calculate Mean and Standard Error (SEM) for each policy
    grouped = df.groupby('policy')[['avg_sim_pathwise', 'avg_sim_reinforce']].agg(['mean', 'sem'])

    policies = grouped.index
    x = np.arange(len(policies))
    width = 0.35

    ax.bar(
        x - width/2,
        grouped['avg_sim_pathwise']['mean'],
        width,
        yerr=grouped['avg_sim_pathwise']['sem'],
        label='Pathwise (B=1)',
        capsize=5,
        color='skyblue',
        edgecolor='black'
    )

    ax.bar(
        x + width/2,
        grouped['avg_sim_reinforce']['mean'],
        width,
        yerr=grouped['avg_sim_reinforce']['sem'],
        label='REINFORCE (B=1000)',
        capsize=5,
        color='salmon',
        edgecolor='black'
    )

    ax.set_title(f"Criss Cross (rho={rho})")
    ax.set_xticks(x)
    ax.set_xticklabels(policies, rotation=0)
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.grid(True, axis='y', alpha=0.25)

# Common labels/legend
for ax in axes[:]:
    if ax.has_data():
        ax.set_ylabel('Cosine Similarity with Ground Truth')

handles, labels = None, None
for ax in axes:
    if ax.has_data():
        handles, labels = ax.get_legend_handles_labels()
        break
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)

fig.suptitle('Gradient Estimator Quality (Criss Cross)', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# 5.2

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
# Section 5.2 
## Fig 9.1

with open("/user/xz3355/QueueTorchReviews/cmu/pathwise_wc_cmu_multiclass5_all_eps.json", "r") as f:
    pw5 = json.load(f)
    
with open("/user/xz3355/QueueTorchReviews/cmu/wc_reinforce_baseline_cmu_B100_multiclass5_all_eps.json", "r") as f:
    rf5 = json.load(f)
    
alphas = ["0.01", "0.1", "0.5", "1.0"]
gaps = ['0.01'] # [1, 0.5, 0.1, 0.05, 0.01]
n = 5
pw5_results = {}

pw5_count = 0
rf5_results = {}
rf5_count = 0

for i in range(n):
    pw5_results[i] = []
    rf5_results[i] = []
    
for alpha in alphas:
    for gap in gaps:
        for x in pw5[alpha][gap]:
            pw5_count += 1
            for i in range(n):
                pw5_results[i].append(x['last_iterate'][0][i])
        for x in rf5[alpha][gap]:
            rf5_count += 1
            for i in range(n):
                rf5_results[i].append(x['last_iterate'][0][i])

print('pw5:', pw5_results)
print('pw5_count:', pw5_count)
pw5_avg_results = [sum(x)/pw5_count for x in pw5_results.values()]
rf5_avg_results = [sum(x)/rf5_count for x in rf5_results.values()]

print(f"Average Policy Score for PATHWISE: {pw5_avg_results}")
print(f"Average Policy Score for REINFORCE: {rf5_avg_results}")

x = np.arange(n)      
width = 0.35              

plt.bar(x - width/2, pw5_avg_results, width, label='PATHWISE(B=1)')
plt.bar(x + width/2, rf5_avg_results, width, label='REINFORCE(B=100)')

plt.title("Epsilon = 0.01")
plt.xticks(x, ['1', '2', '3', '4', '5'])
plt.xlabel('Queue')
plt.ylabel('Policy Score θ_j')
plt.legend()

plt.tight_layout()
plt.show() 
# plt.savefig("/user/xz3355/QueueTorchReviews/cmu/Fig9_1.png")



In [ ]:
## Fig9.2
alphas = [0.01, 0.1, 0.5, 1.0]
gaps = [1, 0.5, 0.1, 0.05, 0.01]

# with open("/user/xz3355/queue-learning/cmu/pathwise_wc_cmu_multiclass.json", 'r') as f:
with open("/user/xz3355/QueueTorchReviews/cmu/pathwise_wc_cmu1_multiclass10.json", "r") as f:
    pw10 = json.load(f)
   
# with open("/user/xz3355/queue-learning/cmu/wc_reinforce_baseline_cmu_multiclass.json", "r") as f: 
with open("/user/xz3355/QueueTorchReviews/cmu/wc_reinforce_baseline_cmu_B10_multiclass10.json", "r") as f:
    rf10 = json.load(f)

pw10_results = []
rf10_results = []
pw10_count = 0
rf10_count = 0

for alpha in alphas:
    for gap in gaps:
        pw10_cost_list = []
        rf10_cost_list = []
        
        for x in pw10[str(alpha)][str(gap)]:
            pw10_count += 1
            pw10_cost_list.append(x['avg_cost'])
            
        for x in rf10[str(alpha)][str(gap)]:
            rf10_count += 1
            rf10_cost_list.append(x['avg_cost'])

        pw10_results.append({'alpha':alpha, 'gap': gap, 'avg_cost': np.mean(pw10_cost_list), 'cost_std': 1.96*np.std(pw10_cost_list)})
        rf10_results.append({'alpha':alpha, 'gap': gap, 'avg_cost': np.mean(rf10_cost_list), 'cost_std': 1.96*np.std(rf10_cost_list)/10})


fig, ax = plt.subplots(figsize=(7.5, 4.5))
x = list(range(len(gaps))) 
gaps = sorted(gaps, reverse=False)

# ===== 1. Pathwise =====
pw_by_alpha = defaultdict(list)
for r in pw10_results:
    pw_by_alpha[r['alpha']].append(r)

for alpha, records in pw_by_alpha.items():
    records = sorted(records, key=lambda x: x['gap'], reverse=False)

    # gaps  = [r['gap'] for r in records]
    means = [r['avg_cost'] for r in records]
    stds  = [r['cost_std'] for r in records]

    ax.errorbar(
        x, means, yerr=stds,
        marker='s',
        linestyle='-',
        linewidth=2.5,
        capsize=4,
        label=f"Pathwise (B=1): α={alpha}"
    )

# ===== 2. REINFORCE =====
rf_by_alpha = defaultdict(list)
for r in rf10_results:
    rf_by_alpha[r['alpha']].append(r)

for alpha, records in rf_by_alpha.items():
    records = sorted(records, key=lambda x: x['gap'], reverse=False)

    # gaps  = [r['gap'] for r in records]
    means = [r['avg_cost'] for r in records]
    stds  = [r['cost_std'] for r in records]

    ax.errorbar(
        x, means, yerr=stds,
        marker='o',
        linestyle='--',
        linewidth=2,
        capsize=4,
        alpha=0.85,
        label=f"REINFORCE + Value (B=10): α={alpha}"
    )

# ===== 3. graph style =====
ax.set_xticks(x)
ax.set_xticklabels(gaps)
ax.set_xlabel("Gap size ε")
ax.set_ylabel("Holding cost of the avg iterate")
ax.invert_xaxis()
ax.grid(True, alpha=0.3)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()
# plt.savefig("/user/xz3355/QueueTorchReviews/cmu/Fig9_2.png")

# 5.3

In [ ]:
# Load data for admission control summary
summary_path = '../results/admission_control_like_buffer_summary.json'

with open(summary_path, 'r') as f:
    summary_data = json.load(f)

In [ ]:
# Extract data for plotting
# We want to plot Mean Cost vs Number of Servers (or Hyper-parameter index)
# The keys are like "reentrant_2_hyper.yaml", "reentrant_3_hyper.yaml", etc.
# We'll extract the number (2, 3, 4...) as the x-axis.

# Separate data into two groups: "reentrant" and "re-reentrant"
reentrant_data = {}
rereentrant_data = {}

for key, value in summary_data.items():
    # Parse key to get type and index
    parts = key.split('_')
    # parts[0] is 'reentrant' or 're-reentrant' (actually 're-reentrant' splits to ['re-reentrant', ...])
    # Let's handle carefully.

    if key.startswith('re-reentrant'):
        idx = int(parts[1][0])*3 # 're-reentrant_2_hyper.yaml' -> parts=['re-reentrant', '2', 'hyper.yaml']
        rereentrant_data[idx] = value
    elif key.startswith('reentrant'):
        idx = int(parts[1][0])*3 # 'reentrant_2_hyper.yaml' -> parts=['reentrant', '2', 'hyper.yaml']
        reentrant_data[idx] = value

# Function to prepare dataframe for plotting
def prepare_df(data_dict):
    rows = []
    for idx, methods in data_dict.items():
        for method, stats in methods.items():
            rows.append({
                'Index': idx,
                'Method': method,
                'Mean': stats['mean'],
                'Std': stats['std']
            })
    return pd.DataFrame(rows).sort_values('Index')

df_re = prepare_df(reentrant_data)
df_rere = prepare_df(rereentrant_data)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

# Define markers/styles for different methods if needed, or rely on seaborn defaults
markers = {'PATHWISE_B1': 'o', 'SPSA_B10': 's', 'SPSA_B100': '^', 'SPSA_B1000': 'D'}

# Plot Reentrant
sns.lineplot(data=df_re, x='Index', y='Mean', hue='Method', style='Method', markers=True, ax=axes[0])
axes[0].set_title('Reentrant Line')
axes[0].set_xlabel('Number of classes')
axes[0].set_ylabel('Mean Cost')

# Plot Re-reentrant
sns.lineplot(data=df_rere, x='Index', y='Mean', hue='Method', style='Method', markers=True, ax=axes[1])
axes[1].set_title('Re-reentrant Line')
axes[1].set_xlabel('Number of classes')
axes[1].set_ylabel('') # Shared Y axis

plt.tight_layout()
plt.show()


# Section 6

In [ ]:
import matplotlib.pyplot as plt
import json

with open("/user/xz3355/QueueTorchReviews/PPO/WC_results.json", "r") as f:
    ppo_wc = json.load(f)
    
with open("/user/xz3355/QueueTorchReviews/PPO/vanilla_bc_results.json", "r") as f:
    ppo_bc = json.load(f)
    
with open("/user/xz3355/QueueTorchReviews/PPO/vanilla_results.json", "r") as f:
    ppo_vanilla = json.load(f)
    
with open("/user/xz3355/QueueTorchReviews/PPO/cmu_results.json", "r") as f:
    cmu_rule = json.load(f)
    
# x axis（policy iterations）
x = range(len(ppo_wc['test_cost']))

plt.figure(figsize=(7.2, 4.6), dpi=120)


plt.plot(x, ppo_wc['test_cost'], linewidth=3.5, label="PPO-WC (B = 20)")
plt.plot(x, ppo_bc['test_cost'], linewidth=3.5, label="PPO-BC (B = 20)")
plt.plot(x, ppo_vanilla['test_cost'], linewidth=3.5, label="PPO Vanilla (B = 50)")
plt.plot(x, [cmu_rule['reentrant_2']['avg_cost']]*len(ppo_wc['test_cost']), linewidth=2.5, label=r"$c\mu$ rule")

# y_axis
plt.yscale("log")
plt.ylim(9, 3e3)

plt.xlabel("Policy Iterations", fontsize=16)
plt.ylabel("Average Holding Cost", fontsize=16)

plt.legend(
    loc="center",
    bbox_to_anchor=(0.68, 0.52),
    frameon=True,
    framealpha=0.9,
    fontsize=12
)

plt.tight_layout()
plt.show()
